In [ ]:
%load_ext autoreload
%autoreload 2

import cProfile
import pstats

import matplotlib.pyplot as plt
import numpy as np

from vetuner.correction import sample_correction_factors, select_samples
from vetuner.drive_cycle import generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.gating import gate_samples
from vetuner.performance import (
    arithmetic_intensity,
    estimate_flops,
    nearest_cell_indices_fast,
    peak_memory_bytes,
    time_call,
)
from vetuner.sensors import apply_sensor_model
from vetuner.surface_fit import SURFACE_METHODS, cell_sample_counts, fit_surface, nearest_cell_indices
from vetuner.ve_surface import BASE_MAP_SURFACE, ve_table

train = generate_drive_cycle()
base = ve_table(params=BASE_MAP_SURFACE)
sim = simulate_engine(train, base)
log = apply_sensor_model(sim, seed=0)
kept = select_samples(log, gate_samples(log).accepted)
factors = sample_correction_factors(kept)
counts = cell_sample_counts(kept.rpm, kept.map_kpa)

n_samples, n_cells, n_visited = len(kept), counts.size, int((counts > 0).sum())
print(f"{n_samples} samples, {n_visited} of {n_cells} cells visited")

In [ ]:
profiler = cProfile.Profile()
profiler.enable()
for _ in range(20):
    fit_surface("per_cell", factors, kept.rpm, kept.map_kpa)
profiler.disable()

pstats.Stats(profiler).sort_stats("cumulative").print_stats(12)

In [ ]:
print(f"{'method':18s} {'best ms':>9} {'noise':>7} {'MFLOP':>9} {'GFLOP/s':>9} {'peak MB':>9}")

measured = {}
for method in SURFACE_METHODS:
    timing = time_call(
        fit_surface, method, factors, kept.rpm, kept.map_kpa, repeats=5
    )
    _, peak = peak_memory_bytes(fit_surface, method, factors, kept.rpm, kept.map_kpa)
    flops = estimate_flops(method, n_samples, n_cells, n_visited)
    measured[method] = timing.best_ms

    print(f"{method:18s} {timing.best_ms:9.2f} {timing.noise_ratio:7.2f} "
          f"{flops / 1e6:9.1f} {flops / (timing.best_ms * 1e6):9.2f} {peak / 1e6:9.2f}")

In [ ]:
for n in (5_000, 50_000, 500_000):
    rng = np.random.default_rng(0)
    rpm = rng.uniform(800.0, 7000.0, n)
    map_kpa = rng.uniform(20.0, 100.0, n)

    slow = time_call(nearest_cell_indices, rpm, map_kpa, repeats=3)
    fast = time_call(nearest_cell_indices_fast, rpm, map_kpa, repeats=3)
    _, slow_mem = peak_memory_bytes(nearest_cell_indices, rpm, map_kpa)
    _, fast_mem = peak_memory_bytes(nearest_cell_indices_fast, rpm, map_kpa)

    print(f"n = {n:>7}   broadcast {slow.best_ms:7.2f} ms / {slow_mem / 1e6:6.1f} MB   "
          f"searchsorted {fast.best_ms:7.2f} ms / {fast_mem / 1e6:6.1f} MB   "
          f"speedup {slow.best_ms / fast.best_ms:.1f}x")

In [ ]:
sizes = (500, 1_000, 2_500, 5_000, 10_000, 25_000)
fig, ax = plt.subplots(figsize=(8, 5))

for method in ("per_cell", "bilinear", "thin_plate", "gaussian_process"):
    times = []
    for n in sizes:
        index = np.linspace(0, len(factors) - 1, min(n, len(factors))).astype(int)
        timing = time_call(
            fit_surface, method, factors[index], kept.rpm[index], kept.map_kpa[index],
            repeats=3,
        )
        times.append(timing.best_ms)
    ax.loglog(sizes[: len(times)], times, marker="o", ms=4, label=method)

ax.set_xlabel("samples")
ax.set_ylabel("fit time (ms)")
ax.set_title("Cost scaling with log length")
ax.legend()
ax.grid(alpha=0.3, which="both")
fig.tight_layout()